# Inverse PINN for Five-Parameter Identification in the Malaria Transmission PDE System

This notebook implements a **fully inverse Physics-Informed Neural Network (PINN)** for the malaria transmission PDE system, where the five parameters
\[
(d, v, \Lambda, \phi, f_T)
\]
are learned simultaneously, while the duration parameters remain fixed.

## Objective

The goal is to identify:
- the diffusion coefficient \(d\),
- the advection coefficient \(v\),
- the transmission rate \(\Lambda\),
- the clinical progression probability \(\phi\),
- the treatment coverage \(f_T\),

from the numerical simulation data generated by the reference solver.

## Main ingredients of this implementation

This notebook includes:
- a data-only initialization strategy for the reaction parameters,
- robust OLS estimation for \((d,v)\),
- data-only estimation of the reaction matrix with EMA stabilization,
- structure-aware Huber losses and focused matrix penalties,
- adaptive phase-dependent training weights,
- uncertainty-based multi-task balancing,
- curvature-aware collocation,
- stronger smoothing in the exposure variable,
- a soft \(H^1\) regularization term,
- diffusion/advection identifiability diagnostics.

## Global setup and imports

This cell initializes the environment, loads the required libraries, and fixes the random seeds for reproducibility.

In [ ]:
import os, math, random, time, json
import numpy as np
import torch, torch.nn as nn
import torch.nn.functional as F

SEED = 2025
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# PyTorch determinism (may raise errors if a non-deterministic op is used)
torch.use_deterministic_algorithms(True)

## Configuration

This cell defines the data path, the advection sign convention, the report-only reference values, the fixed epidemiological durations, and the admissible ranges for the learned reaction parameters.

In [ ]:
NPZ_PATH = "malaria_simulation_results.npz"  # place the file here
assert os.path.isfile(NPZ_PATH), "Upload 'malaria_simulation_results.npz'"

# Sign convention for advection:
# PDE: Ut + Ua - d Uxx + v Ux - U A^T = 0  -> SIGN_V = +1
# PDE: Ut + Ua - d Uxx - v Ux - U A^T = 0  -> SIGN_V = -1
SIGN_V = -1

# True values (for reporting only; they do not influence training)
TRUE_FOR_REPORT = {
    "d": 0.005,
    "v": 0.060,
    "Lambda": 0.35,
    "phi": 0.40,
    "f_T": 0.50,
}

# Fixed durations (known parameters)
KNOWN_FIXED = {"d_D": 5.0, "d_T": 5.0, "d_A": 110.0, "d_P": 25.0}

# Ranges of the learned parameters (Lambda, phi, f_T)
RANGES_A3 = {"Lambda": (0.20, 0.55), "phi": (0.20, 0.75), "f_T": (0.20, 0.8)}

# --- Globals for A_ols EMA ---
A_ols_ema = None
EMA_BETA  = 0.9

## Data loading and basic statistics

This cell loads the numerical solution, computes the coordinate bounds, standardizes the outputs component-wise, reconstructs the inflow function \(b(t)\), and extracts the initial snapshot.

In [ ]:
Z = np.load(NPZ_PATH, allow_pickle=True, mmap_mode='r')
U_snaps = Z["U_snaps"]   # (Nt, Na, Nx, 5)
times   = Z["times"]     # (Nt,)
age_grid= Z["age_grid"]  # (Na,)
xi_grid = Z["xi_grid"]   # (Nx,)
mu_true = Z["mu_true"]  # [d_true, v_true] (if present in the NPZ file)
print("U_snaps:", U_snaps.shape, "| mu_true (NPZ):", mu_true.tolist())

t_min, t_max = float(times.min()), float(times.max())
a_min, a_max = float(age_grid.min()), float(age_grid.max())
x_min, x_max = float(xi_grid.min()), float(xi_grid.max())

# Output statistics (component-wise standardization), RAM-friendly
it = np.linspace(0, len(times)-1, min(20, len(times)), dtype=int)
ia = np.linspace(0, len(age_grid)-1, min(40, len(age_grid)), dtype=int)
ix = np.linspace(0, len(xi_grid)-1, min(40, len(xi_grid)), dtype=int)
buf = np.concatenate([U_snaps[i][np.ix_(ia, ix)].reshape(-1,5) for i in it], axis=0)
y_mean = torch.tensor(buf.mean(0), dtype=torch.float32, device=device)
y_std  = torch.tensor(buf.std(0)+1e-6, dtype=torch.float32, device=device)

# Component-wise weighting to balance the 5 channels in the losses
comp_w = (1.0 / (y_std + 1e-6)).to(device)   # (5,)

def b_from_npz(tval: float):
    if tval <= times[0]: return float(U_snaps[0,0,:,0].mean())
    if tval >= times[-1]: return float(U_snaps[-1,0,:,0].mean())
    i = max(0, min(len(times)-2, int(np.searchsorted(times, tval)-1)))
    w = (tval-times[i])/(times[i+1]-times[i]+1e-12)
    return float((1-w)*U_snaps[i,0,:,0].mean() + w*U_snaps[i+1,0,:,0].mean())

phi0_np = U_snaps[0]  # snapshot at t=0

d_true_npz = float(mu_true[0])
v_true_npz = float(mu_true[1])
print(f"NPZ truth: d={d_true_npz:.6f}, v={v_true_npz:.6f}")
print(f"Report truth (config): {TRUE_FOR_REPORT}")

## Local finite-difference derivatives from data

This cell computes local derivatives using five-point finite-difference formulas, together with stronger smoothing along the exposure direction to stabilize second derivatives.

In [ ]:
def _sample_interior_indices(n, margin=2):
    Nt, Na, Nx, _ = U_snaps.shape
    ti = np.random.randint(margin, Nt-margin, size=n)
    ai = np.random.randint(margin, Na-margin, size=n)
    xi = np.random.randint(margin, Nx-margin, size=n)
    return ti, ai, xi

def _five_point_first(m2, m1, c0, p1, p2, h):
    return (-p2 + 8*p1 - 8*m1 + m2) / (12.0*h)

def _five_point_second(m2, m1, c0, p1, p2, h):
    return (-p2 + 16*p1 - 30*c0 + 16*m1 - m2) / (12.0*h*h)

def _smooth_along_x(arr_1d, w=9):
    if w <= 1: return arr_1d
    pad = w//2
    pad_left  = arr_1d[:pad][::-1]
    pad_right = arr_1d[-pad:][::-1]
    ext = np.concatenate([pad_left, arr_1d, pad_right], axis=0)
    ker = np.ones(w, dtype=np.float64)/w
    sm = np.convolve(ext, ker, mode='valid')
    return sm[:len(arr_1d)]

def _data_local_derivs(n_samples):
    Nt, Na, Nx, C = U_snaps.shape
    ti, ai, xi = _sample_interior_indices(n_samples, margin=2)

    dt = float(times[1]-times[0]) if Nt>1 else 1.0
    da = float(age_grid[1]-age_grid[0]) if Na>1 else 1.0
    dx = float(xi_grid[1]-xi_grid[0]) if Nx>1 else 1.0

    Ut_list, Ua_list, Ux_list, Uxx_list, U_list = [], [], [], [], []
    for k in range(n_samples):
        t0, a0, x0 = ti[k], ai[k], xi[k]

        Ut_m2 = U_snaps[t0-2, a0,   x0,   :]
        Ut_m1 = U_snaps[t0-1, a0,   x0,   :]
        Ut_0  = U_snaps[t0,   a0,   x0,   :]
        Ut_p1 = U_snaps[t0+1, a0,   x0,   :]
        Ut_p2 = U_snaps[t0+2, a0,   x0,   :]

        Ua_m2 = U_snaps[t0,   a0-2, x0,   :]
        Ua_m1 = U_snaps[t0,   a0-1, x0,   :]
        Ua_0  = U_snaps[t0,   a0,   x0,   :]
        Ua_p1 = U_snaps[t0,   a0+1, x0,   :]
        Ua_p2 = U_snaps[t0,   a0+2, x0,   :]

        line_x   = U_snaps[t0, a0, :, :]
        line_x_s = np.stack([_smooth_along_x(line_x[:,c], w=9) for c in range(5)], axis=1)
        Ux_m2 = line_x_s[x0-2, :]; Ux_m1 = line_x_s[x0-1, :]; Ux_0 = line_x_s[x0, :]
        Ux_p1 = line_x_s[x0+1, :]; Ux_p2 = line_x_s[x0+2, :]

        Ut_k  = _five_point_first (Ut_m2, Ut_m1, Ut_0, Ut_p1, Ut_p2, dt)
        Ua_k  = _five_point_first (Ua_m2, Ua_m1, Ua_0, Ua_p1, Ua_p2, da)
        Ux_k  = _five_point_first (Ux_m2, Ux_m1, Ux_0, Ux_p1, Ux_p2, dx)
        Uxx_k = _five_point_second(Ux_m2, Ux_m1, Ux_0, Ux_p1, Ux_p2, dx)

        Ut_list.append(Ut_k); Ua_list.append(Ua_k); Ux_list.append(Ux_k); Uxx_list.append(Uxx_k)
        U_list.append(Ux_0)

    Ut_t  = torch.tensor(np.stack(Ut_list,0),  dtype=torch.float32, device=device)
    Ua_t  = torch.tensor(np.stack(Ua_list,0),  dtype=torch.float32, device=device)
    Ux_t  = torch.tensor(np.stack(Ux_list,0),  dtype=torch.float32, device=device)
    Uxx_t = torch.tensor(np.stack(Uxx_list,0), dtype=torch.float32, device=device)
    U_t   = torch.tensor(np.stack(U_list,0),   dtype=torch.float32, device=device)
    return Ut_t, Ua_t, Ux_t, Uxx_t, U_t

## Data-only OLS estimators and structural losses

This cell defines robust OLS estimators for \((d,v)\), a data-only OLS estimate of the reaction matrix, and the structural penalty terms used during inverse training.

In [ ]:
# ===========
# CELL 5 — OLS & data-only losses (d,v) + A_ols (data-only) + priors
# ===========
@torch.no_grad()
def theta_ols_from_data(batch_n=12000, ridge=2e-4, A_use=None):
    Ut, Ua, Ux, Uxx, U = _data_local_derivs(batch_n)
    if A_use is None:
        raise ValueError("theta_ols_from_data requires A_use to avoid circular dependencies.")
    S = Ut + Ua - (U @ A_use.T)
    d_list, v_list = [], []
    I2 = torch.eye(2, device=device)
    for k in range(5):
        X = torch.stack([Uxx[:,k], -(SIGN_V * Ux[:,k])], dim=1)
        Y = S[:,k:k+1]
        mu = X.mean(0, keepdim=True); sg = X.std(0, keepdim=True) + 1e-8
        Xn = (X - mu)/sg
        beta_n = torch.linalg.solve(Xn.T @ Xn + ridge*I2, Xn.T @ Y)
        beta = beta_n / sg.T
        d_list.append(beta[0,0].item()); v_list.append(beta[1,0].item())
    def iqr_med(v):
        q1, q3 = np.percentile(v, [25, 75]); iqr=q3-q1; lo,hi=q1-1.5*iqr, q3+1.5*iqr
        vv=[x for x in v if lo<=x<=hi]; vv=vv if len(vv)>=1 else v
        return float(np.median(vv))
    return iqr_med(d_list), iqr_med(v_list)

@torch.no_grad()
def A_ols_from_data(d_use, v_use, batch_n=8192, ridge=5e-5, max_tries=4):
    """
    Robust OLS (Tikhonov) for A (5x5), assuming (d_use, v_use) are known.
    - Solves min_W || U_s W - RHS ||^2 + ridge ||W||^2 via an augmented system,
      where U_s = U / std(U) (predictor scaling only).
    - Returns A_ols in the original scale (not in the scaled space).
    - Uses float64 for better stability.
    """
    # 1) Samples and RHS
    Ut, Ua, Ux, Uxx, U = _data_local_derivs(batch_n)
    dtype = torch.float64
    U   = U.to(dtype)
    Ut  = Ut.to(dtype); Ua = Ua.to(dtype); Ux = Ux.to(dtype); Uxx = Uxx.to(dtype)

    RHS = Ut + Ua - d_use*Uxx + SIGN_V*v_use*Ux   # (N,5) float64

    # 2) Predictor scaling only
    eps = torch.tensor(1e-12, dtype=dtype, device=U.device)
    stdU = U.std(dim=0, keepdim=True).clamp_min(1e-8)  # (1,5)
    U_s  = U / stdU                                    # (N,5)

    # For Tikhonov: solve [U_s; sqrt(ridge)*I] w = [RHS; 0]
    I5 = torch.eye(5, dtype=dtype, device=U.device)
    zero5 = torch.zeros(5, dtype=dtype, device=U.device)

    # 3) Column-wise least-squares solve with lstsq (more stable than solve(XtX, XtY))
    W_scaled = torch.empty((5,5), dtype=dtype, device=U.device)  # columns = outputs k
    for k in range(5):
        y = RHS[:, k]

        this_ridge = ridge
        sol = None
        for _ in range(max_tries):
            U_aug = torch.vstack([U_s, (this_ridge**0.5) * I5])              # (N+5, 5)
            y_aug = torch.hstack([y, zero5])                                  # (N+5,)
            # lstsq: least-squares solution (handles rank deficiency)
            sol = torch.linalg.lstsq(U_aug, y_aug).solution                   # (5,)
            if torch.isfinite(sol).all():
                break
            this_ridge *= 10.0  # adaptive ridge if numerical issue occurs

        if sol is None or not torch.isfinite(sol).all():
            # Ultimate fallback: pseudo-inverse (SVD)
            U_pinv = torch.linalg.pinv(U_aug)
            sol = U_pinv @ y_aug

        # sol is the solution in the "scaled" space (U_s)
        # Projection back to the original scale:
        # U_s = U / stdU => U_s * w_scaled ≈ RHS  -  U * (w_scaled / stdU) = RHS
        W_scaled[:, k] = sol

    # 4) Projection back to the original scale (divide by stdU row-wise)
    A_ols_T = (W_scaled / stdU.squeeze(0))  # (5,5) columns = k
    A_ols   = A_ols_T.T                     # (5,5) consistent with "U @ A^T = RHS"

    return A_ols.to(torch.float32)

def huber(x, delta=0.05):
    a = x.abs()
    quad = torch.minimum(a, torch.tensor(delta, device=x.device))
    lin  = a - quad
    return 0.5*(quad**2)/delta + lin

def A_ols_focus_huber(A_cur, A_ols, delta=0.01, w_focus=4.0):
    D = A_cur - A_ols
    absD = D.abs()
    L_h = torch.where(absD <= delta, 0.5*(absD**2), delta*(absD - 0.5*delta)).mean()
    key = [(0,0),(1,0),(2,0),(3,3)]
    Lf = 0.0
    for (i,j) in key:
        dij = absD[i,j]
        lij = (0.5*dij**2) if dij <= delta else (delta*(dij - 0.5*delta))
        Lf = Lf + lij
    Lf = Lf / len(key)
    return L_h + w_focus*Lf

def A_focus_loss(A_cur, A_tgt, w_entries=(1.0,1.0,1.0,0.5), delta=0.05):
    Ls = []
    Ls.append(w_entries[0]*huber(A_cur[0,0]-A_tgt[0,0], delta).mean())
    Ls.append(w_entries[1]*huber(A_cur[1,0]-A_tgt[1,0], delta).mean())
    Ls.append(w_entries[2]*huber(A_cur[2,0]-A_tgt[2,0], delta).mean())
    Ls.append(w_entries[3]*huber(A_cur[3,3]-A_tgt[3,3], delta).mean())
    return sum(Ls)

def theta_ols_prior_loss(Wd=1200.0, Wv=300.0, batch_n=12000, A_use=None):
    d_ols, v_ols = theta_ols_from_data(batch_n=batch_n, A_use=A_use)
    d, v = theta_learn()
    return Wd * (d - d_ols)**2 + Wv * (v - v_ols)**2

## Learnable inverse parameters

This cell defines the constrained parametrizations used to learn the full inverse parameter set \((d, v, \Lambda, \phi, f_T)\).

In [ ]:
def _inv_sig_from_val(val, lo, hi):
    x = float(np.clip((val-lo)/(hi-lo), 1e-6, 1-1e-6))
    return math.log(x/(1-x))

class LearnTheta(nn.Module):
    def __init__(self, d_lo=1e-4, d_hi=5e-2, v_lo=-2e-1, v_hi=2e-1, d_init=0.004, v_init=0.05):
        super().__init__()
        self.d_lo, self.d_hi = d_lo, d_hi
        self.v_lo, self.v_hi = v_lo, v_hi
        self.r_d = nn.Parameter(torch.tensor(_inv_sig_from_val(d_init, d_lo, d_hi), dtype=torch.float32))
        self.r_v = nn.Parameter(torch.tensor(_inv_sig_from_val(v_init, v_lo, v_hi), dtype=torch.float32))
    def forward(self):
        d = self.d_lo + (self.d_hi - self.d_lo) * torch.sigmoid(self.r_d)
        v = self.v_lo + (self.v_hi - self.v_lo) * torch.sigmoid(self.r_v)
        return d, v

class LearnA3(nn.Module):
    def __init__(self, p_init, ranges):
        super().__init__()
        self.keys = ["Lambda","phi","f_T"]
        rho=[]
        for k in self.keys:
            lo,hi = ranges[k]
            rho.append(_inv_sig_from_val(p_init[k], lo,hi))
        self.rho = nn.Parameter(torch.tensor(rho, dtype=torch.float32))
        self.lo  = torch.tensor([ranges[k][0] for k in self.keys], device=device)
        self.hi  = torch.tensor([ranges[k][1] for k in self.keys], device=device)
    def forward(self):
        s = torch.sigmoid(self.rho.to(device))
        p = self.lo + (self.hi - self.lo)*s
        return {k: p[i] for i,k in enumerate(self.keys)}

def A_from_parts(learned, fixed):
    La, ph, fT = learned["Lambda"], learned["phi"], learned["f_T"]
    dD, dT, dA, dP = [torch.tensor(fixed[k], device=device) for k in ["d_D","d_T","d_A","d_P"]]
    A = torch.zeros(5,5,device=device)
    A[0,0]=-La; A[0,3]=1/dA; A[0,4]=1/dP
    A[1,0]=ph*(1-fT)*La; A[1,1]=-1/dD; A[1,3]=ph*(1-fT)*La
    A[2,0]=ph*fT*La; A[2,2]=-1/dT; A[2,3]=ph*fT*La
    A[3,0]=(1-ph)*La; A[3,1]=1/dD; A[3,3]=-(ph*La+1/dA)
    A[4,2]=1/dT; A[4,4]=-1/dP
    return A

## Neural network architecture

This cell defines the Fourier-feature encoding and the MLP used by the inverse PINN.

In [ ]:
class FourierFeatures(nn.Module):
    def __init__(self,K=4): super().__init__(); self.K=K
    def forward(self,z):
        feats=[z]
        for f in range(1,self.K+1):
            w=2*math.pi*f; feats += [torch.sin(w*z), torch.cos(w*z)]
        return torch.cat(feats,1)

class EncodedMLP(nn.Module):
    def __init__(self, width=128, depth=5, act="tanh", K=4):
        super().__init__()
        self.enc=FourierFeatures(K)
        in_dim=3*(1+2*K)
        actl={"tanh":nn.Tanh(),"relu":nn.ReLU(),"silu":nn.SiLU()}[act]
        layers=[nn.Linear(in_dim,width),actl]
        for _ in range(depth-1): layers+=[nn.Linear(width,width),actl]
        layers+=[nn.Linear(width,5)]
        self.net=nn.Sequential(*layers)
    def forward(self,x):
        t_,a_,x_ = x[:, :1], x[:,1:2], x[:,2:3]
        return self.net(torch.cat([self.enc(t_), self.enc(a_), self.enc(x_)],1))

model = EncodedMLP(128,5,"tanh",K=4).to(device)
def predict_std(t_, a_, x_): return model(torch.cat([t_, a_, x_], 1))

## Sampling utilities

This cell defines the batches used for supervised data fitting, initial-condition enforcement, inflow enforcement, and physics collocation.

In [ ]:
def pick_grid(n):
    it=np.random.randint(0,len(times),size=n)
    ia=np.random.randint(0,len(age_grid),size=n)
    ix=np.random.randint(0,len(xi_grid),size=n)
    t=torch.tensor((times[it]-t_min)/(t_max-t_min+1e-12),dtype=torch.float32,device=device).reshape(-1,1)
    a=torch.tensor((age_grid[ia]-a_min)/(a_max-a_min+1e-12),dtype=torch.float32,device=device).reshape(-1,1)
    x=torch.tensor((xi_grid[ix]-x_min)/(x_max-x_min+1e-12),dtype=torch.float32,device=device).reshape(-1,1)
    U=torch.tensor(U_snaps[it,ia,ix,:],dtype=torch.float32,device=device)
    return t,a,x,(U-y_mean)/y_std

def batch_IC(n):
    ia=np.random.randint(0,len(age_grid),size=n)
    ix=np.random.randint(0,len(xi_grid),size=n)
    t0=torch.zeros(n,1,device=device)
    a=torch.tensor((age_grid[ia]-a_min)/(a_max-a_min+1e-12),dtype=torch.float32,device=device).reshape(-1,1)
    x=torch.tensor((xi_grid[ix]-x_min)/(x_max-x_min+1e-12),dtype=torch.float32,device=device).reshape(-1,1)
    U=torch.tensor(phi0_np[ia,ix,:],dtype=torch.float32,device=device)
    return t0,a,x,(U-y_mean)/y_std

def batch_A0(n):
    t=torch.rand(n,1,device=device); a0=torch.zeros_like(t); x=torch.rand(n,1,device=device)
    t_phys=(t*(t_max-t_min)+t_min).flatten().tolist()
    b_vals=torch.tensor([b_from_npz(tt) for tt in t_phys],dtype=torch.float32,device=device).reshape(-1,1)
    U=torch.cat([b_vals, torch.zeros_like(b_vals).repeat(1,4)],1)
    return t,a0,x,(U-y_mean)/y_std

def batch_phys(n, edge_bias=0.6):
    nb=int(n*edge_bias); nu=n-nb
    def beta_edge(m):
        z=torch.distributions.Beta(0.5,0.5).sample((m,1)).to(device); return z.clamp(0,1)
    t_b=beta_edge(nb); a_b=beta_edge(nb)
    k=nb//2; x_b=torch.cat([torch.zeros(k,1,device=device), torch.ones(nb-k,1,device=device)],0)
    t_u=torch.rand(nu,1,device=device); a_u=torch.rand(nu,1,device=device); x_u=torch.rand(nu,1,device=device)
    t=torch.cat([t_b,t_u],0); a=torch.cat([a_b,a_u],0); x=torch.cat([x_b,x_u],0)
    idx=torch.randperm(n,device=device)
    return t[idx],a[idx],x[idx]

def batch_phys_curvature_aware(n, pool=4, edge_bias=0.8):
    m = n*pool
    t,a,x = batch_phys(m, edge_bias=edge_bias)
    eps = 1.0 / max(1, (len(xi_grid)-1))
    with torch.no_grad():
        U0  = model(torch.cat([t, a, x.clamp(0,1)], 1))
        Up  = model(torch.cat([t, a, (x+eps).clamp(0,1)], 1))
        Um  = model(torch.cat([t, a, (x-eps).clamp(0,1)], 1))
        Uxx = (Up - 2*U0 + Um) / (eps**2)
        Uxx = Uxx * (y_std / ((x_max-x_min+1e-12)**2))
        score = Uxx.abs().mean(dim=1)
    idx = torch.topk(score, k=n, largest=True).indices
    return t[idx], a[idx], x[idx]

## PDE-on-data and second-derivative matching losses

This cell defines the data-driven PDE loss and the \(U_{xx}\) matching loss used to stabilize inverse estimation.

In [ ]:
def pde_on_data_loss(n_samples=4096):
    Ut_t, Ua_t, Ux_t, Uxx_t, U_t = _data_local_derivs(n_samples)
    d, v = theta_learn()
    A    = A_from_parts(A3_learn(), KNOWN_FIXED)
    rhs  = Ut_t + Ua_t - d*Uxx_t + SIGN_V * v * Ux_t
    lhs  = U_t @ A.T
    res  = (lhs - rhs) * comp_w
    return (res.pow(2).mean())

def uxx_match_loss(n_samples=4096):
    _, _, _, Uxx_data, _ = _data_local_derivs(n_samples)
    t = torch.rand(n_samples,1,device=device)
    a = torch.rand(n_samples,1,device=device)
    x = torch.rand(n_samples,1,device=device)
    eps = 1.0 / max(1, (len(xi_grid)-1))
    Up  = model(torch.cat([t, a, (x+eps).clamp(0,1)], 1))
    U0  = model(torch.cat([t, a, x], 1))
    Um  = model(torch.cat([t, a, (x-eps).clamp(0,1)], 1))
    x_rng=(x_max-x_min)+1e-12
    Uxx_mod = (Up - 2*U0 + Um) / (eps**2) * (y_std / (x_rng**2))
    diff = (Uxx_mod - Uxx_data.to(device)) * comp_w
    return (diff**2).mean()

## Physics scaling and auxiliary penalties

This cell computes channel-wise physics scales, defines the normalized PDE residual, and introduces the soft \(H^1\) regularization term.

In [ ]:
@torch.no_grad()
def compute_physics_scales_inv(n0: int = 4000):
    t = torch.rand(n0,1,device=device); a = torch.rand(n0,1,device=device); x = torch.rand(n0,1,device=device)
    with torch.enable_grad():
        t_rng=(t_max-t_min)+1e-12; a_rng=(a_max-a_min)+1e-12; x_rng=(x_max-x_min)+1e-12
        t1=t.clone().requires_grad_(True); a1=a; x1=x
        Ustd_t=model(torch.cat([t1,a1,x1],1))
        Ut=torch.cat([torch.autograd.grad(Ustd_t[:,k].sum(), t1, create_graph=True, retain_graph=(k<4))[0] for k in range(5)],1)*(y_std/t_rng)
        t2=t; a2=a.clone().requires_grad_(True); x2=x
        Ustd_a=model(torch.cat([t2,a2,x2],1))
        Ua=torch.cat([torch.autograd.grad(Ustd_a[:,k].sum(), a2, create_graph=True, retain_graph=(k<4))[0] for k in range(5)],1)*(y_std/a_rng)
        t3=t; a3=a; x3=x
        Ustd_c=model(torch.cat([t3,a3,x3],1))
        U=Ustd_c*y_std+y_mean
        eps=1.0/max(1,(len(xi_grid)-1))
        Up=model(torch.cat([t3,a3,(x3+eps).clamp(0,1)],1))
        Um=model(torch.cat([t3,a3,(x3-eps).clamp(0,1)],1))
        Ux =(Up-Um)/(2*eps)*(y_std/x_rng)
        Uxx=(Up-2*Ustd_c+Um)/(eps**2)*(y_std/(x_rng**2))
    AU = U @ A_from_parts(A3_learn(), KNOWN_FIXED).T
    def med(v): return torch.quantile(v.abs(), 0.5, dim=0).clamp_min(1e-6)
    return {"Ut":med(Ut), "Ua":med(Ua), "Ux":med(Ux), "Uxx":med(Uxx), "AU":med(AU)}

def pde_residual_only_scaled(t_, a_, x_, scales):
    d, v = theta_learn()
    A = A_from_parts(A3_learn(), KNOWN_FIXED)
    t1=t_.clone().requires_grad_(True); a1=a_.detach(); x1=x_.detach()
    Ustd_t=model(torch.cat([t1,a1,x1],1))
    Ut = torch.cat([torch.autograd.grad(Ustd_t[:,k].sum(), t1, create_graph=True, retain_graph=True)[0] for k in range(5)], 1)
    t2=t_.detach(); a2=a_.clone().requires_grad_(True); x2=x_.detach()
    Ustd_a=model(torch.cat([t2,a2,x2],1))
    Ua = torch.cat([torch.autograd.grad(Ustd_a[:,k].sum(), a2, create_graph=True, retain_graph=True)[0] for k in range(5)], 1)
    t3=t_.detach(); a3=a_.detach(); x3=x_.detach()
    Ustd_c=model(torch.cat([t3,a3,x3],1))
    U=Ustd_c*y_std+y_mean
    eps=1.0/max(1,(len(xi_grid)-1))
    Up=model(torch.cat([t3,a3,(x3+eps).clamp(0,1)],1))
    Um=model(torch.cat([t3,a3,(x3-eps).clamp(0,1)],1))
    x_rng=(x_max-x_min)+1e-12
    Ux =(Up-Um)/(2*eps)*(y_std/x_rng)
    Uxx=(Up-2*Ustd_c+Um)/(eps**2)*(y_std/(x_rng**2))
    r = (Ut/scales["Ut"]) + (Ua/scales["Ua"]) - d*(Uxx/scales["Uxx"]) + SIGN_V*v*(Ux/scales["Ux"]) - (U @ A.T)/scales["AU"]
    return r * comp_w

def h1_penalty(n=2048):
    t=torch.rand(n,1,device=device); a=torch.rand(n,1,device=device); x=torch.rand(n,1,device=device).requires_grad_(True)
    Ustd = model(torch.cat([t,a,x],1))
    dUdx=[]
    for k in range(5):
        gk = torch.autograd.grad(Ustd[:,k].sum(), x, create_graph=False, retain_graph=True)[0]
        dUdx.append(gk)
    dUdx=torch.cat(dUdx,1)
    x_rng=(x_max-x_min)+1e-12
    dUdx = dUdx * (y_std/x_rng)
    return (dUdx**2).mean()

## Data-only initialization and model instantiation

This cell selects the initial reaction parameter guess from data-only candidates and instantiates the inverse parameter modules.

In [ ]:
def choose_p_init_from_ranges(ranges, n_cands=24, jitter=0.05, batch_n=4000, ridge=2e-4):
    def mid(lo,hi): return 0.5*(lo+hi)
    base = {k: mid(*ranges[k]) for k in ("Lambda","phi","f_T")}
    A_mid = A_from_parts(
        {"Lambda": torch.tensor(base["Lambda"], device=device),
         "phi":    torch.tensor(base["phi"],    device=device),
         "f_T":    torch.tensor(base["f_T"],    device=device)},
        KNOWN_FIXED
    ).detach()
    d_ols, v_ols = theta_ols_from_data(batch_n=12000, ridge=ridge, A_use=A_mid)
    Ut, Ua, Ux, Uxx, U = _data_local_derivs(batch_n)
    RHS = Ut + Ua - d_ols*Uxx + SIGN_V*v_ols*Ux
    cands=[]
    for _ in range(n_cands):
        cand={}
        for k in ("Lambda","phi","f_T"):
            lo,hi = ranges[k]
            span = (hi-lo)
            val  = base[k] + (np.random.rand()-0.5)*(2*jitter*span)
            val  = float(np.clip(val, lo, hi))
            cand[k]=val
        cands.append(cand)
    best=None; best_score=float("inf")
    for c in cands:
        A = A_from_parts(
            {"Lambda": torch.tensor(c["Lambda"],device=device),
             "phi":    torch.tensor(c["phi"],   device=device),
             "f_T":    torch.tensor(c["f_T"],   device=device)},
            KNOWN_FIXED
        )
        res = (U @ A.T) - RHS
        sc  = float((res**2).mean().item())
        if sc < best_score:
            best_score = sc; best = c
    return best

P_INIT = choose_p_init_from_ranges(RANGES_A3, n_cands=24, jitter=0.05)
theta_learn = LearnTheta(d_init=0.004, v_init=0.05).to(device)
A3_learn    = LearnA3(P_INIT, RANGES_A3).to(device)
print("P_INIT:", P_INIT)

## Uncertainty-based multi-task weighting

This cell defines the uncertainty-weighting mechanism used to adaptively balance the inverse loss terms.

In [ ]:
class UncertaintyWeights(nn.Module):
    def __init__(self):
        super().__init__()
        self.names = ["DATA","PDE","IC","A0","FLUX","STRUCT","UXX","PDATA","AFOCUS"]
        self.logsig = nn.ParameterDict({n: nn.Parameter(torch.tensor(0.0,device=device)) for n in self.names})
    def forward(self, losses: dict):
        total = 0.0; eff={}
        for n in self.names:
            if n not in losses: continue
            s=self.logsig[n]; Li=losses[n]
            wi=torch.exp(-s); total = total + wi*Li + s
            eff[n]=float(wi.detach().cpu())
        return total, eff

## Inverse training loop

This cell runs the full inverse optimization procedure, including adaptive OLS scheduling, phase-dependent weighting, curvature-aware collocation, soft parameter freezing, and early stopping.

In [ ]:
BPhys = 4096; BData = 2048; BIC = 1024; BA0 = 4096; BNeu = 4096
EPOCHS = 7000

uw = UncertaintyWeights().to(device)
scales = compute_physics_scales_inv(n0=4000)

# Optimizer (A3 updated more slowly)
params_theta = list(theta_learn.parameters())         # d, v
params_model = list(model.parameters())               # network
params_A     = list(A3_learn.parameters())            # A3

opt = torch.optim.AdamW(
    [
      {'params': params_model, 'lr': 1e-3,  'weight_decay': 1e-4},
      {'params': params_theta,'lr': 8e-4,  'weight_decay': 1e-4},
      {'params': params_A,    'lr': 3e-4,  'weight_decay': 5e-5},  # slower A3
    ]
)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS*2, eta_min=2e-4)

L_H1 = 3e-4

# Slow EMA for A_ols
A_ols_ema = None
EMA_BETA  = 0.985

# Caches to avoid expensive recomputations
A_ols_cache = None
d_ols_cache, v_ols_cache = None, None

print("Training (inverse, OLS schedule + adaptive batches + early stopping)...")
t0 = time.time()

# Phases
P1_end = 3000
P2_end = 6000  # shorter P2 -> time gain, P3 for refinement

def scale_grads_local(module, scale):
    for p in module.parameters():
        if p.grad is not None:
            p.grad.mul_(scale)

def trainable_params_from(optimizer):
    # clip only currently trainable parameters
    return [p for g in optimizer.param_groups for p in g['params'] if getattr(p, 'requires_grad', True)]

def cadence_recomp(ep: int) -> int:
    # OLS recomputation frequency (epochs)
    if ep < 800:   return 2
    if ep < 2000:  return 5
    if ep < 3400:  return 10
    if ep < 5200:  return 15
    return 20

def rel_err_pct(h, t):  # local helper for early stopping
    return float(abs(h - t) / (abs(t) + 1e-12) * 100.0)

ok_count = 0  # counts consecutive checkpoints below 10%

for ep in range(1, EPOCHS+1):

    # Adaptive batches (lower cost after P1)
    if ep <= P1_end:
        BPhys_eff = BPhysժմ= BPhys      # 4096
        BNeu_eff  = BNeu       # 4096
    else:
        BPhys_eff = 3072
        BNeu_eff  = 2048

    # Loss weights / “soft-freeze” (better quality in P1, robustness in P2, refinement in P3)
    edge_bias = 0.95 if ep < 1200 else (0.9 if ep < 3000 else 0.8)
    if ep <= P1_end:
        phase    = "P1"
        W_UXX    = 10.0
        Wd_ols   = 3800.0
        Wv_ols   = 900.0
        W_STRUCT = 0.6
        W_AFOCUS = 0.6
        gT, gA   = 1.0, 1.0
        uxx_ns   = 4096
    elif ep <= P2_end:
        phase    = "P2"
        W_UXX    = 12.0
        Wd_ols   = 1000.0
        Wv_ols   = 300.0
        W_STRUCT = 1.2
        W_AFOCUS = 1.0
        gT, gA   = 1.5, 0.2
        uxx_ns   = 3072
    else:
        phase    = "P3"
        W_UXX    = 5.0
        Wd_ols   = 500.0
        Wv_ols   = 200.0
        W_STRUCT = 0.9
        W_AFOCUS = 0.6
        gT, gA   = 1.0, 1.0
        uxx_ns   = 2048

    # Hard freeze of A3 during P2
    if phase == "P2":
        for p in A3_learn.parameters():
            p.requires_grad = False
            if p.grad is not None:
                p.grad = None
    else:
        for p in A3_learn.parameters():
            p.requires_grad = True

    # OLS schedule + slow EMA
    if (ep % cadence_recomp(ep)) == 1 or (A_ols_cache is None):
        with torch.no_grad():
            A_for_theta = A_from_parts(A3_learn(), KNOWN_FIXED).detach()

            # theta_ols (light)
            d_ols, v_ols = theta_ols_from_data(batch_n=12000, A_use=A_for_theta)
            d_ols_cache, v_ols_cache = d_ols, v_ols

            # A_ols (large batch at the beginning, lighter later)
            if ep < 1200:   bn_A = 8192
            elif ep < 3400: bn_A = 4096
            else:           bn_A = 2048
            A_ols = A_ols_from_data(d_use=d_ols, v_use=v_ols, batch_n=bn_A, ridge=5e-5)
            A_ols_cache = A_ols

            if A_ols_ema is None:
                A_ols_ema = A_ols.clone()
            else:
                A_ols_ema = EMA_BETA * A_ols_ema + (1.0 - EMA_BETA) * A_ols
    else:
        d_ols, v_ols = d_ols_cache, v_ols_cache
        A_ols        = A_ols_cache
        A_for_theta  = A_from_parts(A3_learn(), KNOWN_FIXED).detach()

    # === Forward losses ===
    opt.zero_grad(set_to_none=True)

    # PDE (curvature-aware collocation)
    t_p, a_p, x_p = batch_phys_curvature_aware(BPhys_eff, pool=4, edge_bias=edge_bias)
    r_pde  = pde_residual_only_scaled(t_p, a_p, x_p, scales)
    L_pde  = (r_pde**2).mean()

    # PDE-on-data (anchors d, v, A)
    L_pdata = pde_on_data_loss(n_samples=4096)

    # Robust OLS priors on (d,v)
    def theta_ols_prior_loss_local():
        d, v = theta_learn()
        return Wd_ols * (d - d_ols)**2 + Wv_ols * (v - v_ols)**2
    L_theta = theta_ols_prior_loss_local()

    # Robin flux (x=0,1)
    def robin_flux_loss(BNeu_eff):
        d_cur, v_cur = theta_learn()
        d_det, v_det = d_cur.detach(), v_cur.detach()
        t_n = torch.rand(BNeu_eff,1,device=device)
        a_n = torch.rand(BNeu_eff,1,device=device)
        x0 = torch.zeros(BNeu_eff//2,1,device=device, requires_grad=True)
        x1 = torch.ones (BNeu_eff - BNeu_eff//2,1,device=device, requires_grad=True)
        def flux_loss_at_x(xb, t_slice, a_slice):
            Ustd = model(torch.cat([t_slice, a_slice, xb], 1))
            d_list=[]
            for k in range(5):
                gk = torch.autograd.grad(Ustd[:,k].sum(), xb, create_graph=True, retain_graph=True)[0]
                d_list.append(gk)
            dUdx = torch.cat(d_list, 1) * (y_std / ((x_max - x_min) + 1e-12))
            U_phys = Ustd * y_std + y_mean
            F = -d_det * dUdx + SIGN_V * v_det * U_phys
            return (F**2).mean()
        n0 = x0.shape[0]
        L0 = flux_loss_at_x(x0, t_n[:n0], a_n[:n0])
        L1 = flux_loss_at_x(x1, t_n[n0:], a_n[n0:])
        return 0.5*(L0 + L1)
    L_flux = robin_flux_loss(BNeu_eff)

    # A structure (EMA)
    A_cur    = A_from_parts(A3_learn(), KNOWN_FIXED)
    L_struct = A_ols_focus_huber(A_cur, A_ols_ema, delta=0.01, w_focus=4.0)
    L_afocus = A_focus_loss(A_cur, A_ols_ema, w_entries=(1.0,1.0,1.0,0.5), delta=0.05)

    # Data / IC / A0
    t_d, a_d, x_d, Ud = pick_grid(BData); Up_d  = predict_std(t_d,a_d,x_d); L_data = ((Up_d-Ud)**2).mean()
    t_ic, a_ic, x_ic, U0 = batch_IC(BIC); Up_ic = predict_std(t_ic,a_ic,x_ic); L_ic   = ((Up_ic-U0)**2).mean()
    t_a0, a0, x_a0, Ua   = batch_A0(BA0); Up_a0 = predict_std(t_a0,a0,x_a0); L_a0   = ((Up_a0-Ua)**2).mean()

    # Uxx match with adaptive cost
    L_uxx = uxx_match_loss(n_samples=uxx_ns)

    # H1 smoothing + light regularization
    L_h1  = h1_penalty(n=2048)
    y_rand = model(torch.rand(2048,3,device=device)); L_reg = (y_rand**2).mean()

    losses = dict(
        DATA=L_data, PDE=L_pde, IC=L_ic, A0=L_a0, FLUX=L_flux,
        STRUCT=W_STRUCT*L_struct, UXX=W_UXX*L_uxx, PDATA=L_pdata, AFOCUS=W_AFOCUS*L_afocus
    )
    tot, eff = uw(losses)
    tot = tot + 1e-7*L_reg + L_H1*L_h1 + L_theta
    tot.backward()

    # “Soft-freeze” by gradient scaling
    scale_grads_local(theta_learn, gT)
    scale_grads_local(A3_learn,    gA)

    # Trust-region on A3 (not in P2 because it is frozen)
    max_step = 0.15
    if phase != "P2":
        with torch.no_grad():
            for p in params_A:
                if p.grad is not None:
                    gnorm = p.grad.norm()
                    if torch.isfinite(gnorm) and gnorm > max_step:
                        p.grad.mul_(max_step / (gnorm + 1e-12))

    # Global clipping on currently trainable parameters
    params_trainables = trainable_params_from(opt)
    torch.nn.utils.clip_grad_norm_(params_trainables, 1.0)

    opt.step(); sched.step()

    # Logs + early stopping every 500 epochs
    if (ep % 500) == 0 or ep == 1:
        with torch.no_grad():
            d_hat, v_hat = theta_learn(); pA = A3_learn()
            d_ols_q, v_ols_q = theta_ols_from_data(batch_n=4096, A_use=A_for_theta)
        dt = time.time() - t0; t0 = time.time()
        print(
            f"Ep {ep:4d} | phase={phase} "
            f"| Lpde {L_pde.item():.2e} Lpdata {L_pdata.item():.2e} "
            f"Lstruct {L_struct.item():.2e} Lafocus {L_afocus.item():.2e} "
            f"| d={float(d_hat):.5f} (OLS {d_ols_q:.5f}) v={float(v_hat):.5f} (OLS {v_ols_q:.5f}) "
            f"| Λ={float(pA['Lambda']):.3f} φ={float(pA['phi']):.3f} fT={float(pA['f_T']):.3f} "
            f"| eff(PDE)={eff.get('PDE',float('nan')):.2f} eff(UXX)={eff.get('UXX',float('nan')):.2f} "
            f"eff(STRUCT)={eff.get('STRUCT',float('nan')):.2f} eff(AFOCUS)={eff.get('AFOCUS',float('nan')):.2f} "
        )

        # Early stop: <10% for (d, v, Λ, φ, f_T) at two consecutive checkpoints (after 4200 epochs)
        d_ok   = rel_err_pct(float(d_hat),          TRUE_FOR_REPORT['d'])      < 10.0
        v_ok   = rel_err_pct(float(v_hat),          TRUE_FOR_REPORT['v'])      < 10.0
        La_ok  = rel_err_pct(float(pA['Lambda']),   TRUE_FOR_REPORT['Lambda']) < 10.0
        phi_ok = rel_err_pct(float(pA['phi']),      TRUE_FOR_REPORT['phi'])    < 10.0
        fT_ok  = rel_err_pct(float(pA['f_T']),      TRUE_FOR_REPORT['f_T'])    < 10.0

        if all([d_ok, v_ok, La_ok, phi_ok, fT_ok]):
            ok_count += 1
            if ep >= 4200 and ok_count >= 2:
                print("Early stop: all parameters < 10% for two consecutive checkpoints.")
                break
        else:
            ok_count = 0

## Final diagnostics and reporting

This cell reports the final inverse estimates, computes relative errors with respect to the report values, evaluates the diffusion/advection identifiability ratio, and saves the summary file.

In [ ]:
@torch.no_grad()
def identifiability_score(n=4000):
    t=torch.rand(n,1,device=device); a=torch.rand(n,1,device=device); x=torch.rand(n,1,device=device)
    U0 = model(torch.cat([t,a,x],1))
    eps=1.0/max(1,(len(xi_grid)-1))
    Up = model(torch.cat([t,a,(x+eps).clamp(0,1)],1))
    Um = model(torch.cat([t,a,(x-eps).clamp(0,1)],1))
    x_rng=(x_max-x_min)+1e-12
    Ux  = (Up-Um)/(2*eps)*(y_std/x_rng)
    Uxx = (Up-2*U0+Um)/(eps**2)*(y_std/(x_rng**2))
    num = Uxx.abs().mean().item()
    den = Ux.abs().mean().item()+1e-9
    return num/den

def rel_err_pct(h, t): return float(abs(h - t) / (abs(t) + 1e-12) * 100.0)

d_hat, v_hat = theta_learn(); pA = A3_learn()

print("\n=== Estimates (learned only) ===")
print(f"d_hat     ={float(d_hat):.6f} (npz {d_true_npz:.6f}, report {TRUE_FOR_REPORT['d']:.6f})")
print(f"v_hat     ={float(v_hat):.6f} (npz {v_true_npz:.6f}, report {TRUE_FOR_REPORT['v']:.6f})")
print(f"Lambda_hat={float(pA['Lambda']):.6f} (report {TRUE_FOR_REPORT['Lambda']:.6f})")
print(f"phi_hat   ={float(pA['phi']):.6f} (report {TRUE_FOR_REPORT['phi']:.6f})")
print(f"f_T_hat   ={float(pA['f_T']):.6f} (report {TRUE_FOR_REPORT['f_T']:.6f})")

print("\n=== Relative errors (%) (vs REPORT truths) ===")
print(f"d:      {rel_err_pct(float(d_hat), TRUE_FOR_REPORT['d']):6.2f}%")
print(f"v:      {rel_err_pct(float(v_hat), TRUE_FOR_REPORT['v']):6.2f}%")
print(f"Lambda: {rel_err_pct(float(pA['Lambda']), TRUE_FOR_REPORT['Lambda']):6.2f}%")
print(f"phi:    {rel_err_pct(float(pA['phi']),    TRUE_FOR_REPORT['phi']):6.2f}%")
print(f"f_T:    {rel_err_pct(float(pA['f_T']),    TRUE_FOR_REPORT['f_T']):6.2f}%")

print("\nIdentifiability (diffusion/advection) ratio ~", identifiability_score())

summary = {
    "SIGN_V": SIGN_V,
    "d_hat": float(d_hat), "v_hat": float(v_hat),
    "Lambda_hat": float(pA["Lambda"]), "phi_hat": float(pA["phi"]), "f_T_hat": float(pA["f_T"]),
    "report_truths": TRUE_FOR_REPORT,
    "npz_truths": {"d": float(d_true_npz), "v": float(v_true_npz)},
}
with open("inverse_summary.json","w") as f:
    json.dump(summary, f, indent=2)
print("\nSaved: inverse_summary.json")